# Illuminator

**Illuminator** analyzes a single true PDF and produces:

1. An **annotated PDF** with colored bounding boxes and global item numbers
2. A **`Table_of_content.md`** listing every detected content item per page

Both files are packaged in **`Output.zip`**.

### Content types detected

| Type | Description |
|------|-------------|
| Title | Document / chapter titles |
| Subtitle | Level-1 section headers |
| Subsubtitle | Level-2+ section headers |
| Text | Individual paragraphs |
| Table | Tabular regions |
| Picture | Images and figures |
| Page number | Footer page numbers |
| Footnote | Footnote text |

### Tools

- **Docling** — primary layout analysis and reading order
- **Camelot (stream)** — table bounding-box refinement
- **PyMuPDF** — page orientation, annotation drawing, heuristics

## 1. Environment setup (Google Colab)

Run the cells below once per session.

> **Note:** The first run downloads Docling layout models (~hundreds of MB) and may take several minutes.

In [ ]:
# Install system dependency for Camelot (Colab / Debian-based Linux)
import shutil
import subprocess
import sys

if shutil.which("gs") is None:
    try:
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ghostscript"], check=False)
    except FileNotFoundError:
        print("apt-get not available — install Ghostscript manually if Camelot fails.")
else:
    print("Ghostscript already available.")

In [ ]:
%pip install -q docling pymupdf "camelot-py[cv]" opencv-python-headless

# Colab ships a newer `typer-slim` than Docling pins, which triggers a harmless
# pip "dependency conflict" warning. Align `typer-slim` to the installed `typer`
# version to clear it. This never raises if the packages are missing.
import importlib.metadata as _md
import subprocess as _sp
import sys as _sys

try:
    _typer_v = _md.version("typer")
    if _md.version("typer-slim") != _typer_v:
        _sp.run(
            [_sys.executable, "-m", "pip", "install", "-q", f"typer-slim=={_typer_v}"],
            check=False,
        )
        print(f"Aligned typer-slim to typer=={_typer_v}.")
    else:
        print("typer / typer-slim versions already aligned.")
except Exception as _exc:
    print(f"typer alignment skipped (safe to ignore): {_exc}")

## 2. Imports and constants

In [ ]:
from __future__ import annotations

import io
import logging
import re
import zipfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import camelot
import fitz  # PyMuPDF
from docling.document_converter import DocumentConverter
from docling_core.types.doc import (
    ContentLayer,
    CoordOrigin,
    PictureItem,
    SectionHeaderItem,
    TableItem,
    TextItem,
    TitleItem,
)
from docling_core.types.doc.labels import DocItemLabel

logging.getLogger("docling").setLevel(logging.WARNING)
logging.getLogger("camelot").setLevel(logging.WARNING)

APP_NAME = "Lumindra"
APP_VERSION = "2.0.0"

CONTENT_TYPES: List[str] = [
    "Title",
    "Subtitle",
    "Subsubtitle",
    "Text",
    "Table",
    "Picture",
    "Page number",
    "Footnote",
]

# RGB colors in 0..1 range for PyMuPDF
TYPE_COLORS: Dict[str, Tuple[float, float, float]] = {
    "Title": (1.0, 0.0, 0.0),        # Red
    "Subtitle": (1.0, 0.55, 0.0),    # Orange
    "Subsubtitle": (0.85, 0.65, 0.13),  # Gold
    "Text": (0.0, 0.0, 1.0),         # Blue
    "Table": (0.0, 0.6, 0.0),       # Green
    "Picture": (0.55, 0.0, 0.55),   # Purple
    "Page number": (0.5, 0.5, 0.5), # Gray
    "Footnote": (0.0, 0.55, 0.55),  # Teal
}

SOURCE_PRIORITY = {"camelot": 3, "docling": 2, "heuristic": 1}
IOU_DEDUP_THRESHOLD = 0.5
BOTTOM_BAND_RATIO = 0.08

## 3. Data models and coordinate helpers

Docling PDF provenance bboxes are normally in **bottom-left** origin; Camelot also uses **bottom-left** origin; PyMuPDF draws in **top-left** page coordinates. Each source is normalized to top-left and scaled to the PyMuPDF page size before annotation.

In [ ]:
@dataclass
class RawItem:
    content_type: str
    page_index: int  # 0-based
    bbox: fitz.Rect
    source: str
    reading_order: float = 0.0


@dataclass
class ContentItem:
    global_no: int = 0
    content_type: str = ""
    type_no: int = 0
    page_index: int = 0
    bbox: fitz.Rect = field(default_factory=lambda: fitz.Rect(0, 0, 0, 0))
    source: str = ""


def docling_bbox_to_fitz(
    bbox: Any,
    dl_width: float,
    dl_height: float,
    page: fitz.Page,
) -> fitz.Rect:
    """Convert a Docling provenance BoundingBox to PyMuPDF (TOPLEFT) coordinates.

    Docling PDF provenance bboxes are usually in BOTTOMLEFT origin. We first
    normalize to top-left origin using the Docling page height, then scale to
    the PyMuPDF page size in case the two coordinate spaces differ.
    """
    try:
        tl = bbox.to_top_left_origin(page_height=dl_height)
        left, top, right, bottom = float(tl.l), float(tl.t), float(tl.r), float(tl.b)
    except Exception:
        origin = getattr(bbox, "coord_origin", None)
        is_bottom_left = (origin == CoordOrigin.BOTTOMLEFT) or (
            str(origin).upper().endswith("BOTTOMLEFT")
        )
        if is_bottom_left:
            top = dl_height - float(bbox.t)
            bottom = dl_height - float(bbox.b)
        else:
            top = float(bbox.t)
            bottom = float(bbox.b)
        left, right = float(bbox.l), float(bbox.r)

    top, bottom = min(top, bottom), max(top, bottom)
    left, right = min(left, right), max(left, right)

    sx = (page.rect.width / dl_width) if dl_width else 1.0
    sy = (page.rect.height / dl_height) if dl_height else 1.0
    return fitz.Rect(left * sx, top * sy, right * sx, bottom * sy)


def camelot_bbox_to_fitz(bbox: Tuple[float, ...], page_height: float) -> fitz.Rect:
    """Convert Camelot _bbox [left, bottom, right, top] (BOTTOMLEFT) to fitz.Rect."""
    left, bottom, right, top = bbox
    return fitz.Rect(left, page_height - top, right, page_height - bottom)


def rect_iou(a: fitz.Rect, b: fitz.Rect) -> float:
    inter = a & b
    if inter.is_empty:
        return 0.0
    inter_area = inter.get_area()
    union_area = a.get_area() + b.get_area() - inter_area
    return inter_area / union_area if union_area > 0 else 0.0


def page_orientation_label(page: fitz.Page) -> str:
    w, h = page.rect.width, page.rect.height
    if page.rotation in (90, 270):
        w, h = h, w
    return "horizontal" if w > h else "vertical"


def sort_key_for_item(item: RawItem | ContentItem) -> Tuple[int, float, float]:
    return (item.page_index, item.bbox.y0, item.bbox.x0)

## 4. Docling extraction

In [ ]:
def classify_docling_label(item: Any) -> Optional[str]:
    label = getattr(item, "label", None)
    if label is None:
        return None

    if isinstance(item, TitleItem) or label == DocItemLabel.TITLE:
        return "Title"
    if isinstance(item, SectionHeaderItem) or label == DocItemLabel.SECTION_HEADER:
        level = getattr(item, "level", 1) or 1
        return "Subtitle" if level <= 1 else "Subsubtitle"
    if isinstance(item, TableItem) or label == DocItemLabel.TABLE:
        return "Table"
    if isinstance(item, PictureItem) or label == DocItemLabel.PICTURE:
        return "Picture"
    if label == DocItemLabel.FOOTNOTE:
        return "Footnote"
    if label == DocItemLabel.PAGE_FOOTER:
        return "Page number"
    if label in (
        DocItemLabel.PARAGRAPH,
        DocItemLabel.TEXT,
        DocItemLabel.LIST_ITEM,
        DocItemLabel.CAPTION,
        DocItemLabel.CODE,
        DocItemLabel.FORMULA,
    ):
        return "Text"
    return None


def docling_page_sizes(doc: Any) -> Dict[int, Tuple[float, float]]:
    sizes: Dict[int, Tuple[float, float]] = {}
    for page_no, page_item in (getattr(doc, "pages", {}) or {}).items():
        size = getattr(page_item, "size", None)
        if size is not None:
            sizes[int(page_no)] = (float(size.width), float(size.height))
    return sizes


def iterate_doc_items(doc: Any):
    """Iterate over both body and furniture so footnotes / page footers are seen."""
    try:
        layers = {ContentLayer.BODY, ContentLayer.FURNITURE}
        return list(doc.iterate_items(included_content_layers=layers))
    except TypeError:
        return list(doc.iterate_items())


def map_docling_item(
    item: Any,
    order_idx: float,
    page_sizes: Dict[int, Tuple[float, float]],
    fitz_doc: fitz.Document,
    page_count: int,
) -> List[RawItem]:
    content_type = classify_docling_label(item)
    if content_type is None:
        return []

    prov_list = getattr(item, "prov", None) or []
    if not prov_list:
        return []

    mapped: List[RawItem] = []
    for prov in prov_list:
        if prov.bbox is None:
            continue
        page_index = prov.page_no - 1
        if not (0 <= page_index < page_count):
            continue
        page = fitz_doc[page_index]
        dl_w, dl_h = page_sizes.get(prov.page_no, (page.rect.width, page.rect.height))
        bbox = docling_bbox_to_fitz(prov.bbox, dl_w, dl_h, page)
        if bbox.is_empty or bbox.get_area() <= 0:
            continue
        mapped.append(
            RawItem(
                content_type=content_type,
                page_index=page_index,
                bbox=bbox,
                source="docling",
                reading_order=order_idx,
            )
        )
    return mapped


def extract_docling_items(doc: Any, fitz_doc: fitz.Document, page_count: int) -> List[RawItem]:
    page_sizes = docling_page_sizes(doc)
    items: List[RawItem] = []
    for order_idx, (item, _level) in enumerate(iterate_doc_items(doc)):
        items.extend(
            map_docling_item(item, float(order_idx), page_sizes, fitz_doc, page_count)
        )
    return items

## 5. Camelot table extraction (stream flavor)

In [ ]:
def extract_camelot_tables(pdf_path: str, page_index: int, page_height: float) -> List[RawItem]:
    page_no = page_index + 1
    try:
        tables = camelot.read_pdf(
            pdf_path,
            pages=str(page_no),
            flavor="stream",
            suppress_stdout=True,
        )
    except Exception as exc:
        print(f"Camelot warning on page {page_no}: {exc}")
        return []

    items: List[RawItem] = []
    for idx, table in enumerate(tables):
        bbox = camelot_bbox_to_fitz(table._bbox, page_height)
        if bbox.is_empty or bbox.get_area() <= 0:
            continue
        items.append(
            RawItem(
                content_type="Table",
                page_index=page_index,
                bbox=bbox,
                source="camelot",
                reading_order=1000 + idx,
            )
        )
    return items

## 6. PyMuPDF heuristics (page numbers and footnotes)

In [ ]:
def _page_has_type(items: List[RawItem], page_index: int, content_type: str) -> bool:
    return any(i.page_index == page_index and i.content_type == content_type for i in items)


def detect_page_number_footnote(page: fitz.Page, page_index: int, existing: List[RawItem]) -> List[RawItem]:
    found: List[RawItem] = []
    page_h = page.rect.height
    bottom_y = page_h * (1.0 - BOTTOM_BAND_RATIO)

    blocks = page.get_text("dict").get("blocks", [])
    candidates: List[Tuple[str, fitz.Rect, float]] = []

    for block in blocks:
        if block.get("type") != 0:
            continue
        for line in block.get("lines", []):
            for span in line.get("spans", []):
                text = (span.get("text") or "").strip()
                if not text:
                    continue
                x0, y0, x1, y1 = span["bbox"]
                if y0 < bottom_y:
                    continue
                rect = fitz.Rect(x0, y0, x1, y1)
                size = float(span.get("size", 10))
                candidates.append((text, rect, size))

    if not _page_has_type(existing, page_index, "Page number"):
        for text, rect, _size in candidates:
            if re.fullmatch(r"\d{1,4}", text):
                found.append(
                    RawItem(
                        content_type="Page number",
                        page_index=page_index,
                        bbox=rect,
                        source="heuristic",
                        reading_order=9000,
                    )
                )
                break

    if not _page_has_type(existing, page_index, "Footnote"):
        footnote_candidates = [
            (text, rect, size)
            for text, rect, size in candidates
            if not re.fullmatch(r"\d{1,4}", text)
        ]
        if footnote_candidates:
            text, rect, size = min(footnote_candidates, key=lambda c: c[2])
            if size <= 11 or re.match(r"^\d+[\.)\s]", text):
                found.append(
                    RawItem(
                        content_type="Footnote",
                        page_index=page_index,
                        bbox=rect,
                        source="heuristic",
                        reading_order=9100,
                    )
                )

    return found

## 7. Deduplication and numbering

In [ ]:
def classify_and_deduplicate(raw_items: List[RawItem]) -> List[RawItem]:
    kept: List[RawItem] = []

    for item in sorted(raw_items, key=lambda r: (-SOURCE_PRIORITY.get(r.source, 0), sort_key_for_item(r))):
        replaced = False
        for idx, existing in enumerate(kept):
            if existing.page_index != item.page_index:
                continue
            if existing.content_type != item.content_type:
                continue
            if rect_iou(existing.bbox, item.bbox) >= IOU_DEDUP_THRESHOLD:
                if SOURCE_PRIORITY.get(item.source, 0) > SOURCE_PRIORITY.get(existing.source, 0):
                    kept[idx] = item
                replaced = True
                break
        if not replaced:
            kept.append(item)

    return sorted(kept, key=sort_key_for_item)


def assign_numbers(items: List[RawItem]) -> List[ContentItem]:
    type_counters: Dict[str, int] = {t: 0 for t in CONTENT_TYPES}
    global_no = 0
    result: List[ContentItem] = []

    for raw in items:
        global_no += 1
        type_counters[raw.content_type] += 1
        result.append(
            ContentItem(
                global_no=global_no,
                content_type=raw.content_type,
                type_no=type_counters[raw.content_type],
                page_index=raw.page_index,
                bbox=raw.bbox,
                source=raw.source,
            )
        )
    return result

## 8. PDF annotation, TOC, and ZIP output

In [ ]:
def annotate_pdf(doc: fitz.Document, items: List[ContentItem]) -> bytes:
    label_size = 8.0
    for item in items:
        page = doc[item.page_index]
        color = TYPE_COLORS.get(item.content_type, (0, 0, 0))

        # Clamp the box to the page so nothing is drawn off-canvas.
        box = item.bbox & page.rect
        if box.is_empty:
            box = item.bbox

        # Colored border only — no fill.
        page.draw_rect(
            box,
            color=color,
            fill=None,
            width=1.2,
            overlay=True,
        )

        # Item number at the top-left of the box, in the same color.
        label = str(item.global_no)
        label_y = box.y0 - 1
        if label_y - label_size < 0:
            label_y = box.y0 + label_size + 1  # inside the box if near the top edge
        label_point = fitz.Point(box.x0 + 0.5, label_y)
        page.insert_text(
            label_point,
            label,
            fontsize=label_size,
            color=color,
            render_mode=0,
        )

    buf = io.BytesIO()
    doc.save(buf, garbage=4, deflate=True)
    return buf.getvalue()


def render_toc(
    items: List[ContentItem],
    page_count: int,
    input_name: str,
    orientations: Dict[int, str],
) -> str:
    lines = [
        f"# {APP_NAME} v{APP_VERSION} — Table of Content",
        "",
        f"- Input: `{input_name}`",
        f"- Pages: {page_count}",
        "",
        "## Contents",
        "",
    ]

    by_page: Dict[int, List[ContentItem]] = {p: [] for p in range(page_count)}
    for item in items:
        by_page[item.page_index].append(item)

    for page_index in range(page_count):
        orient = orientations.get(page_index, "unknown")
        lines.append(f"- Page {page_index + 1} ({orient}):")
        page_items = sorted(by_page[page_index], key=lambda i: (i.bbox.y0, i.bbox.x0))
        for item in page_items:
            lines.append(
                f"  - No. {item.global_no}, {item.content_type} (No. {item.type_no}),"
            )
        lines.append("")

    return "\n".join(lines).rstrip() + "\n"


def build_output_zip(annotated_pdf_bytes: bytes, toc_md: str, zip_path: str = "Output.zip") -> str:
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("annotated.pdf", annotated_pdf_bytes)
        zf.writestr("Table_of_content.md", toc_md)
    return zip_path

## 9. Main pipeline

In [ ]:
def run_lumindra(pdf_path: str, output_zip: str = "Output.zip") -> str:
    pdf_path = str(Path(pdf_path).resolve())
    if not Path(pdf_path).is_file():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    doc_fitz = fitz.open(pdf_path)
    page_count = doc_fitz.page_count
    if page_count == 0:
        raise ValueError("PDF has no pages.")

    orientations = {i: page_orientation_label(doc_fitz[i]) for i in range(page_count)}

    print(f"Converting with Docling ({page_count} pages)...")
    converter = DocumentConverter()
    dl_result = converter.convert(pdf_path)
    dl_doc = dl_result.document

    raw_items: List[RawItem] = extract_docling_items(dl_doc, doc_fitz, page_count)

    docling_table_pages = {i.page_index for i in raw_items if i.content_type == "Table"}
    pages_for_camelot = docling_table_pages if docling_table_pages else set(range(page_count))

    for page_index in sorted(pages_for_camelot):
        page_h = doc_fitz[page_index].rect.height
        raw_items.extend(extract_camelot_tables(pdf_path, page_index, page_h))

    for page_index in range(page_count):
        raw_items.extend(detect_page_number_footnote(doc_fitz[page_index], page_index, raw_items))

    deduped = classify_and_deduplicate(raw_items)
    numbered = assign_numbers(deduped)

    if not numbered:
        print("Warning: no content items detected. Output will contain empty TOC.")

    annotated_bytes = annotate_pdf(doc_fitz, numbered)
    doc_fitz.close()

    toc = render_toc(numbered, page_count, Path(pdf_path).name, orientations)
    zip_path = build_output_zip(annotated_bytes, toc, output_zip)

    print(f"Done — {len(numbered)} items detected.")
    print(f"Output written to: {zip_path}")
    return zip_path

## 10. Color legend

| Content type | Color |
|--------------|-------|
| Title | Red |
| Subtitle | Orange |
| Subsubtitle | Gold |
| Text | Blue |
| Table | Green |
| Picture | Purple |
| Page number | Gray |
| Footnote | Teal |

## 11. Run on Google Colab

Upload a PDF, run Lumindra, and download **`Output.zip`**.

In [ ]:
try:
    from google.colab import files

    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")

    input_pdf = next(iter(uploaded.keys()))
    zip_path = run_lumindra(input_pdf)
    files.download(zip_path)
except ImportError:
    # Local Jupyter fallback — set INPUT_PDF to your file path
    INPUT_PDF = "sample.pdf"
    zip_path = run_lumindra(INPUT_PDF)
    print(f"Created {zip_path} (local mode — set INPUT_PDF above)")